# Bônus — contrato de dados (`movies`)
Um YAML só (`config/contracts/movies_contract.yaml`) gera tanto os checks em Spark quanto o validator do MongoDB. Roda depois do `01_run_pipeline`, porque precisa da tabela `bronze.sample_mflix__movies` já existir.

In [0]:
import sys
import json
sys.path.append("../src")

from pyspark.sql import functions as F
from pyspark.sql.types import ArrayType, DoubleType, IntegerType, StringType, StructField, StructType

from ingestion.contract import apply_checks, generate_mongo_validator, generate_spark_checks, load_contract

contrato = load_contract("../config/contracts/movies_contract.yaml")
checks = generate_spark_checks(contrato)
for nome, expr in checks:
    print(f"{nome:24} -> {expr}")

## O validator equivalente pro MongoDB

In [0]:
print(json.dumps(generate_mongo_validator(contrato), indent=2, ensure_ascii=False))

## Rodando os checks nos dados reais
A Bronze guarda o JSON cru, então o parse aqui é só pra validar — não muda como a Bronze é gravada.

In [ ]:
schema_movies = StructType([
    StructField("_id", StringType(), True),
    StructField("year", IntegerType(), True),
    StructField("title", StringType(), True),
    StructField("genres", ArrayType(StringType()), True),
    StructField("runtime", IntegerType(), True),
    StructField("imdb", StructType([
        StructField("rating", DoubleType(), True),
    ]), True),
])

df_bronze = spark.table("meu_catalog.bronze.sample_mflix__movies")
df_parsed = (
    df_bronze
    .select("_source_id", F.from_json("body", schema_movies).alias("doc"))
    .select("_source_id", "doc.*")
)

resultados = apply_checks(df_parsed, checks)
spark.createDataFrame(resultados).display()

## Muda o contrato, não o código
Tenta marcar `runtime` como obrigatório no YAML e roda de novo — o resultado muda sem mexer em uma linha de Python.

In [0]:
contrato_recarregado = load_contract("../config/contracts/movies_contract.yaml")
novos_checks = generate_spark_checks(contrato_recarregado)
spark.createDataFrame(apply_checks(df_parsed, novos_checks)).display()